# Hooks and lifecycle

Read the [guide](README.md). Run cells in order with the notebooks and apps groups. Offline replay uses real SDK and local protocol paths; it does not measure live model quality.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_lifecycle_hooks

Predict the outcome, then run the [script](examples/01_lifecycle_hooks.py).


In [ ]:
"""Lifecycle hooks: Nimbus Tier 3 lesson.

Run: uv run python modules/11_hooks/examples/01_lifecycle_hooks.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    audit = AuditPolicy()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": " bos "}),
                "Quote checked against the budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[audit],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: invocation state comes from the application, not model-generated tool arguments.
    result = agent(
        "Quote bos for traveler@example.com",
        invocation_state=dict(budget_cents=50000),
        limits={"turns": 3},
    )
    print("Lifecycle:", audit.events)
    complete(result, cfg)


run_lesson(main, "modules/11_hooks/examples/01_lifecycle_hooks.py")

## 02_tool_hooks

Predict the outcome, then run the [script](examples/02_tool_hooks.py).


In [ ]:
"""Tool hooks: Nimbus Tier 3 lesson.

Run: uv run python modules/11_hooks/examples/02_tool_hooks.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    audit = AuditPolicy()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": " bos "}),
                "Quote checked against the budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[audit],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: invocation state comes from the application, not model-generated tool arguments.
    result = agent(
        "Quote bos for traveler@example.com",
        invocation_state=dict(budget_cents=50000),
        limits={"turns": 3},
    )
    print("Normalized input:", agent.messages[1])
    print("Tool outcomes:", audit.rows)
    complete(result, cfg)


run_lesson(main, "modules/11_hooks/examples/02_tool_hooks.py")

## 03_message_hooks

Predict the outcome, then run the [script](examples/03_message_hooks.py).


In [ ]:
"""Message hooks: Nimbus Tier 3 lesson.

Run: uv run python modules/11_hooks/examples/03_message_hooks.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    audit = AuditPolicy()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": " bos "}),
                "Quote checked against the budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[audit],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: invocation state comes from the application, not model-generated tool arguments.
    result = agent(
        "Quote bos for traveler@example.com",
        invocation_state=dict(budget_cents=50000),
        limits={"turns": 3},
    )
    print("Redacted history:", agent.messages)
    # MessageUpdatedEvent observes framework replacements; plain list edits emit no event.
    from strands.hooks import MessageUpdatedEvent

    audit.observe(
        MessageUpdatedEvent(
            agent=agent,
            tracking_id="fixture",
            message={"role": "user", "content": [{"text": "fixture@example.com"}]},
        )
    )
    print("Explicit replacement-event fixture:", audit.rows[-1])
    complete(result, cfg)


run_lesson(main, "modules/11_hooks/examples/03_message_hooks.py")

## 04_audit_budget

Predict the outcome, then run the [script](examples/04_audit_budget.py).


In [ ]:
"""Audit budget: Nimbus Tier 3 lesson.

Run: uv run python modules/11_hooks/examples/04_audit_budget.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    audit = AuditPolicy()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": "LAX"}),
                "Quote checked against the budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[audit],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: invocation state comes from the application, not model-generated tool arguments.
    result = agent(
        "Quote LAX for traveler@example.com",
        invocation_state=dict(budget_cents=50000),
        limits={"turns": 3},
    )
    print("Trusted budget: $500. Audit:", audit.rows)
    complete(result, cfg)


run_lesson(main, "modules/11_hooks/examples/04_audit_budget.py")

## audited_booking

Predict the outcome, then run the [script](use_case/audited_booking.py).


In [ ]:
"""Ited booking: Nimbus Tier 3 lesson.

Run: uv run python modules/11_hooks/use_case/audited_booking.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    audit = AuditPolicy()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": "LAX"}),
                "Quote checked against the budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[audit],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: invocation state comes from the application, not model-generated tool arguments.
    result = agent(
        "Quote LAX for traveler@example.com",
        invocation_state=dict(budget_cents=50000),
        limits={"turns": 3},
    )
    import json

    from common.config import ROOT

    output = ROOT / ".nimbus/tool_audit.json"
    output.parent.mkdir(exist_ok=True)
    output.write_text(json.dumps(audit.rows, indent=2), encoding="utf-8")
    print("Redacted audit:", output)
    complete(result, cfg)


run_lesson(main, "modules/11_hooks/use_case/audited_booking.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
